# Reranking RAG
### Two-stage retrieval: fast bi-encoder recall, then precise cross-encoder rerank

Corpus: `OWASP Top 10 for LLM Applications (2025)` — 10 named risk categories (LLM01–LLM10) sharing vocabulary like “risk”, “attack”, “model”, which is exactly what makes naive retrieval struggle.

## Step 1: Build the pipeline

In [1]:
!pip install langchain langchain-community langchain-ollama langchain-text-splitters faiss-cpu pypdf sentence-transformers -q


[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_ollama import OllamaEmbeddings, ChatOllama

C:\Users\shiva\AppData\Local\Temp\ipykernel_15840\1805325906.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


C:\Users\shiva\.pyenv\pyenv-win\versions\3.12.10\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
PDF_PATH = "OWASP-Top-10-for-LLMs-v2025.pdf"

pages = PyPDFLoader(PDF_PATH).load()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(pages)

embeddings = OllamaEmbeddings(model="nomic-embed-text:latest")
vector_store = FAISS.from_documents(chunks, embeddings)

llm = ChatOllama(model="llama3.2:3b", temperature=0)

print(f"Loaded {len(pages)} pages -> {len(chunks)} chunks -> {vector_store.index.ntotal} vectors")

incorrect startxref pointer(1)


parsing for Object Streams


Error -3 while decompressing data: invalid code lengths set


Error -3 while decompressing data: invalid code lengths set


Error -3 while decompressing data: invalid code lengths set


Error -3 while decompressing data: invalid code lengths set


Loaded 45 pages -> 131 chunks -> 131 vectors


## Step 2: Stage 1 — broad recall with the bi-encoder
Cast a wide net: retrieve far more candidates than we'll actually use.

In [4]:
query = "What risks come from letting an LLM call external tools and APIs on its own?"

candidates = vector_store.similarity_search(query, k=15)

print("Stage 1 — broad recall (bi-encoder), top 5 of 15 candidates:")
for doc in candidates[:5]:
    print(f"page {doc.metadata['page']}: {doc.page_content[:120]}...")

Stage 1 — broad recall (bi-encoder), top 5 of 15 candidates:
page 14: supply-chain risks. Finally, the emergence of on-device LLMs increase the attack surface and
supply-chain risks for LLM ...
page 30: roles, permission structure of the application) directly in the system prompts. Instead,
externalize such information to...
page 14: OWASP Top 10 for LLM Applications v2.0
11genai.owasp.org
LLM03:2025 Supply Chain
Description
LLM supply chains are susce...
page 27: Follow secure coding best practice, such as applying OWASP’s recommendations in ASVS
(Application Security Verification ...
page 4: Retrieval-Augmented Generation (RAG) and other embedding-based methods, now core practices
for grounding model outputs.
...


## Step 3: Stage 2 — cross-encoder reranks the candidates
The cross-encoder reads the query and each candidate *together*, so it catches relevance the bi-encoder's independent embeddings miss.

In [5]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

pairs = [(query, doc.page_content) for doc in candidates]
scores = reranker.predict(pairs)

reranked = sorted(zip(scores, candidates), key=lambda pair: pair[0], reverse=True)

print("Stage 2 — reranked (cross-encoder), top 5:")
for score, doc in reranked[:5]:
    print(f"score={score:.3f}  page {doc.metadata['page']}: {doc.page_content[:120]}...")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Loading weights:  36%|███▌      | 38/105 [00:00<00:00, 378.09it/s]

Loading weights:  78%|███████▊  | 82/105 [00:00<00:00, 384.36it/s]

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 491.03it/s]

Stage 2 — reranked (cross-encoder), top 5:
score=0.797  page 14: OWASP Top 10 for LLM Applications v2.0
11genai.owasp.org
LLM03:2025 Supply Chain
Description
LLM supply chains are susce...
score=0.531  page 14: supply-chain risks. Finally, the emergence of on-device LLMs increase the attack surface and
supply-chain risks for LLM ...
score=-1.050  page 36: OWASP Top 10 for LLM Applications v2.0
33genai.owasp.org
level of expertise. For example, chatbots have been found to mi...
score=-1.182  page 25: LLM to malfunction. Common triggers include:
• hallucination/confabulation caused by poorly-engineered benign prompts, o...
score=-2.322  page 4: larger, more diverse group of contributors worldwide who have all helped shape this list. The
process involved brainstor...


## Step 4: Compare the ordering
Same 15 candidates, two different rankings — see how much the top 5 shuffles.

In [6]:
bi_encoder_order = [doc.metadata["page"] for doc in candidates[:5]]
cross_encoder_order = [doc.metadata["page"] for _, doc in reranked[:5]]

print(f"Bi-encoder top-5 pages:    {bi_encoder_order}")
print(f"Cross-encoder top-5 pages: {cross_encoder_order}")

Bi-encoder top-5 pages:    [14, 30, 14, 27, 4]
Cross-encoder top-5 pages: [14, 14, 36, 25, 4]


## Step 5: Trim to top-K and generate the final answer

In [7]:
top_k_docs = [doc for _, doc in reranked[:4]]
context = "\n\n".join(doc.page_content for doc in top_k_docs)

prompt = f"""Answer the question based only on the following context:

{context}

Question: {query}
Answer:"""

print(llm.invoke(prompt).content)

Letting an LLM call external tools and APIs on its own can lead to excessive functionality, which is a risk that can impact the confidentiality, integrity, and availability of systems. This is because the LLM may be able to interact with multiple systems and access sensitive information without proper oversight or control, potentially leading to unintended consequences such as data breaches or system compromise.


## Try it yourself
1. Widen Stage 1 to `k=50` — does the final top-5 change?
2. Try `BAAI/bge-reranker-base` in place of the MiniLM cross-encoder.
3. Pick a query where the bi-encoder's #1 result is actually wrong, and confirm the reranker fixes it.